# Chain Model Evaluation - US Traffic Incident Analysis
ในขั้นตอนนี้เราจะนำ Model ทั้งสองตัวที่ Train ไว้มาทดสอบกับ Test Set โดยใช้กระบวนการแบบ Chain:
1. ใช้ **Model 1** ทำนาย `Distance(mi)` จาก Features พื้นฐาน
2. นำ `Distance(mi)` ที่ทำนายได้ ไปรวมกับ Features พื้นฐาน
3. ใช้ **Model 2** ทำนาย `Duration(min)`
4. วัดผลลัพธ์สุดท้ายเปรียบเทียบกับคำตอบจริง (Ground Truth)

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load Test Data & Models

In [2]:
# โหลด Features ของ Test Set
X_test = pd.read_csv("../data/processed/03/test_features.csv")

# โหลดคำตอบจริง (Ground Truth)
y_true = pd.read_csv("../data/processed/01.2/test_secret_answers.csv")

# โหลด Models
model_dist = joblib.load("../models/chain_dist_model.pkl")
model_dur = joblib.load("../models/chain_dur_model.pkl")

print(f"X_test shape: {X_test.shape}")
print(f"y_true shape: {y_true.shape}")

## 2. Step 1: Predict Distance
ใช้ Model 1 ทำนายระยะทาง

In [3]:
print("Predicting Distance...")
pred_distance = model_dist.predict(X_test)

# ตรวจสอบผลลัพธ์ของ Model 1
dist_mae = mean_absolute_error(y_true["Distance(mi)"], pred_distance)
print(f"Distance Prediction MAE: {dist_mae:.4f}")

## 3. Step 2: Predict Duration using Predicted Distance
นำผลลัพธ์จาก Step 1 มาเป็น Feature ให้ Model 2

In [4]:
X_test_with_dist = X_test.copy()
X_test_with_dist["Distance(mi)"] = pred_distance

print("Predicting Duration...")
pred_duration = model_dur.predict(X_test_with_dist)

# ตรวจสอบผลลัพธ์ของ Model 2 (Final Prediction)
dur_mae = mean_absolute_error(y_true["Duration(min)"], pred_duration)
dur_rmse = np.sqrt(mean_squared_error(y_true["Duration(min)"], pred_duration))
dur_r2 = r2_score(y_true["Duration(min)"], pred_duration)

print(f"\n--- Final Evaluation (Duration) ---")
print(f"MAE:  {dur_mae:.4f} minutes")
print(f"RMSE: {dur_rmse:.4f} minutes")
print(f"R2:   {dur_r2:.4f}")

## 4. Visualize Results

In [5]:
plt.figure(figsize=(12, 5))

# Plot Distribution ของผลการทำนาย vs ความจริง
plt.subplot(1, 2, 1)
sns.kdeplot(y_true["Duration(min)"], label='Actual', fill=True)
sns.kdeplot(pred_duration, label='Predicted', fill=True)
plt.title("Duration Distribution: Actual vs Predicted")
plt.legend()

# Scatter Plot แสดงความสัมพันธ์ (สุ่มมา 10,000 จุดเพื่อให้กราฟไม่แน่นเกินไป)
plt.subplot(1, 2, 2)
sample_idx = np.random.choice(len(y_true), 10000, replace=False)
plt.scatter(y_true["Duration(min)"].iloc[sample_idx], pred_duration[sample_idx], alpha=0.3)
plt.plot([y_true["Duration(min)"].min(), y_true["Duration(min)"].max()], 
         [y_true["Duration(min)"].min(), y_true["Duration(min)"].max()], 'r--')
plt.xlabel("Actual Duration")
plt.ylabel("Predicted Duration")
plt.title("Actual vs Predicted Scatter")

plt.tight_layout()
plt.show()

## 5. Error Analysis
ดูว่าอุบัติเหตุประเภทไหนที่ Model ทำนายพลาดมากที่สุด

In [6]:
error_df = X_test.copy()
error_df['Actual_Duration'] = y_true['Duration(min)']
error_df['Pred_Duration'] = pred_duration
error_df['Absolute_Error'] = np.abs(error_df['Actual_Duration'] - error_df['Pred_Duration'])

print("Top 5 states with highest average error:")
if 'State' in error_df.columns:
    print(error_df.groupby('State')['Absolute_Error'].mean().sort_values(ascending=False).head())
else:
    # ถ้า State ถูก Encode ไปแล้ว ลองดูตาม Weather_Group
    print(error_df.groupby('Weather_Group')['Absolute_Error'].mean().sort_values(ascending=False).head())